In [ ]:
nThreads = 10
import os

os.environ["OMP_NUM_THREADS"] = f"{nThreads}"
os.environ["OPENBLAS_NUM_THREADS"] = f"{nThreads}"
os.environ["MKL_NUM_THREADS"] = f"{nThreads}"
os.environ["BLIS_NUM_THREADS"] = f"{nThreads}"
os.environ["VECLIB_MAXIMUM_THREADS"] = f"{nThreads}"
os.environ["MKL_DYNAMIC"] = "FALSE"

In [ ]:
import os

SEED = 1


os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8") 


# -------------------------------------------------------------
# now import libraries
import random
import numpy as np

random.seed(SEED)
np.random.seed(SEED)

# Torch
import torch
torch.manual_seed(SEED)


import scanpy as sc, anndata as ad, numpy as np, pandas as pd
import logging
import triku as tk 
from matplotlib import pylab
import os
import sys
import yaml
from scipy.sparse import csr_matrix
import gc
import torch
import scanit
from scipy.sparse import issparse
import scanit
from scipy.sparse import issparse
import gc
import torch

In [ ]:
sc.set_figure_params(dpi=50, facecolor='white', dpi_save=500)
pylab.rcParams['figure.figsize'] = (6, 6)
homeDir = os.getenv("HOME")

sys.path.insert(1, homeDir+"/utils/")


from PlotPCA_components import *
from AdataSanityCheck import *
from PurgeAdata import *
from spatialUtils import *
from _DEAplots import *
from _plotting import *

import rapids_singlecell as rsc


In [ ]:
with open(homeDir+"/utils/config.yaml", 'r') as f:
    analysis_params = yaml.safe_load(f)["analysisParams"]
print(analysis_params)
DS = "B19-25653_8um"
DSname = "B19-25653"
TifName = "B19v3FLIPPED0607.tif"
base_path = "/data/Spatial_Tx" 
HashesDir = homeDir+"/hashes"



import cupyx.scipy.sparse
import random
from scipy import sparse
import rmm
from rmm.allocators.cupy import rmm_cupy_allocator
import cupy as cp

cp.cuda.set_allocator(rmm_cupy_allocator)

# Load data

In [ ]:


adata = sc.read_h5ad(homeDir+f"/adatas/{DS}_CleanAdata.h5ad")
adata


In [ ]:
import squidpy as sq

In [ ]:
fig, get_selected_df, get_boundaries = interactive_spatial_selector(adata, library_id=f"{DSname}_hires_image")

# Backup/import existing boundaries

In [ ]:
import os, json

os.makedirs(HashesDir, exist_ok=True)
boundaries_file = os.path.join(HashesDir, f"Boundaries_{DS}.json")

if os.path.exists(boundaries_file):
    with open(boundaries_file, "r", encoding="utf-8") as f:
        boundariesDict = json.load(f)
    print(f"Restored previously defined boundaries from {boundaries_file}")
else:
    boundariesDict = get_boundaries()
    with open(boundaries_file, "w", encoding="utf-8") as f:
        json.dump(boundariesDict, f, ensure_ascii=False, indent=2)
    print(f"Computed and saved new boundaries to {boundaries_file}")


In [ ]:
boundariesDict

In [ ]:

adata = SubsetSpatialCoords(adata, boundariesDict)
adata.uns["adata_borders"] = boundariesDict

In [ ]:
adata.X = adata.layers["counts"].copy()
import scipy.sparse as sp
adata.X = sp.csr_matrix(adata.X)




print(adata)
# mitochondrial genes, "MT-" for human, "Mt-" for mouse
adata.var["mt"] = adata.var_names.str.startswith("MT-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes
adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")

sc.pp.calculate_qc_metrics(adata, inplace=True)
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt", "ribo", "hb"],  log1p=True, inplace=True
)



In [ ]:
adata.obs.columns

In [ ]:
for i in ["log1p_total_counts","pct_counts_hb","pct_counts_mt","n_genes_by_counts","log1p_n_genes_by_counts"]:
    sc.pl.violin(adata,keys=[i])


In [ ]:
adata = adata[(adata.obs["pct_counts_mt"] < 20) & (adata.obs["pct_counts_hb"] < 20)].copy()

In [ ]:
adata = adata[adata.obs["n_genes_by_counts"] > 20].copy()

In [ ]:
adata = adata[adata.obs["total_counts"] > 20].copy()
adata

In [ ]:
np.expm1(3)

In [ ]:
adata

# Import tif

In [ ]:
import cupyx
import seaborn as sns
import itertools
def flatten(list_of_lists):
    "Flatten one level of nesting."
    return list(itertools.chain(*list_of_lists))


In [ ]:
img_fullres = import_fullres_image_metadata(
    adata,
    tissue_positions_path = f"/data/{DSname}_outs/binned_outputs/square_008um/spatial/tissue_positions.parquet",
    fullres_tiff_path = f"/data/projects/spatialTX/data/{TifName}",
    scalefactors_json_path = f"/data/{DSname}_outs/binned_outputs/square_008um/spatial/scalefactors_json.json",
    image_key="Full_res_image",
    verbose=True
)


adata.obsm["FULLRESspatial_microns"] = adata.obsm["FULLRESspatial"] * adata.uns["FULLRESspatial"]["Full_res_image"]["scalefactors"]["microns_per_pixel"]
adata.uns["FULLRESspatial_microns"] = adata.uns["FULLRESspatial"]

#sq.gr.spatial_neighbors(adata, spatial_key="FULLRESspatial_microns",delaunay=True,coord_type="generic", n_neighs=20)



# First import spacexr Decon

In [ ]:
RCTD_res = pd.read_csv(homeDir+f"/2.1_spacexr_deconvolution/RCTD_res{DS}/results_{DS}.csv", index_col=0)

In [ ]:
commonBCs = list(set(RCTD_res.index.tolist()).intersection(set(adata.obs_names.tolist())))
RCTD_res = RCTD_res.loc[commonBCs]

adata = adata[commonBCs].copy()

adata.obs = pd.concat([adata.obs, RCTD_res],axis = 1)


In [ ]:
import numpy as np
import pandas as pd

# masks
doublet = adata.obs["spot_class"].isin(["doublet_certain","doublet_uncertain"])
reject  = adata.obs["spot_class"].isin(["reject"])

# helper to join two labels alphabetically, de-duplicated, ignoring NaNs
def join_sorted(a, b):
    vals = [str(a), str(b)]
    vals = [v for v in vals if v and v.lower() != "nan"]
    vals = sorted(set(vals))
    return ",".join(vals) if vals else ""

# make a Series of the pairwise-joined labels
pairs = adata.obs[["first_type", "second_type"]].astype(str).apply(
    lambda s: join_sorted(s.iloc[0], s.iloc[1]), axis=1
)

# start from first_type, then overwrite per mask
spacexr = adata.obs["first_type"].astype(str).copy()
spacexr[doublet] = pairs[doublet]
spacexr[reject]  = "reject"

adata.obs["spacexr"] = spacexr

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming this is your groupby result:
counts = adata.obs.groupby("spacexr").size()

# Convert to DataFrame for seaborn
df = counts.reset_index(name="Count")
df = df.rename(columns={"spacexr": "CellType"})

# Sort by count descending
df = df.sort_values("Count", ascending=False)

# Barplot
plt.figure(figsize=(20,6))
sns.barplot(data=df, x="CellType", y="Count", color="steelblue")
plt.ylabel("Number of cells")
plt.title("Cell counts by spacexR deconvolution")
plt.tight_layout()
sns.despine(offset=10)
plt.xticks(rotation=90)

plt.show()


# Plot some cell types

In [ ]:
obstomap = "spacexr"

[i for i in adata.obs[obstomap].unique().tolist() if "," not in i and i != "reject"]


In [ ]:
# MincellType
# Here we keep either pure nuclei or spurius nuclei as long as those spurius nuclei have at least 100 entries for a given combination of types
print(adata.shape)
#adata = adata[(~adata.obs["spacexr"].str.contains(",")) | (adata.obs["spacexr"].str.contains(",") & adata.obs["spacexr"].isin(adata.obs["spacexr"].value_counts()[adata.obs["spacexr"].value_counts() > 200].index.tolist()))].copy()
adata = adata[adata.obs["spacexr"] != "reject"].copy()
print(adata.shape)


In [ ]:
adata.obs["spacexr"] = adata.obs["spacexr"].astype("category")

# 1) Plot stromal and endo

In [ ]:
obstomap = "spacexr"
groups = [i for i in adata.obs[obstomap].unique().tolist() if i != "Multiplets" and i != "Negative"]
print(groups)
plot_spatial_obs(
    adata, obs=obstomap, width=5, dpi=100,dotscale=.8 ,img_key="hires",groups=["stromal","Endothelialcells"],
    spatial_key='spatial',scale_key="tissue_hires_scalef",library_id= f"{DSname}_hires_image", enforce_pixel_spot_size=False, img_alpha=.2)

# 2) Lymphoids

In [ ]:
obstomap = "spacexr"
groups = [i for i in adata.obs[obstomap].unique().tolist() if i != "Multiplets" and i != "Negative"]
print(groups)
plot_spatial_obs(
    adata, obs=obstomap, width=5, dpi=100,dotscale=.8 ,img_key="hires",groups=['CD8+Tcells', 'CD4+T', 'NKcells','Plasmacells','Bcells'],
    spatial_key='spatial',scale_key="tissue_hires_scalef",library_id= f"{DSname}_hires_image", enforce_pixel_spot_size=False, img_alpha=.2)

# 2) Myeloids

In [ ]:
obstomap = "spacexr"
groups = [i for i in adata.obs[obstomap].unique().tolist() if i != "Multiplets" and i != "Negative"]
print(groups)
plot_spatial_obs(
    adata, obs=obstomap, width=5, dpi=100,dotscale=.8 ,img_key="hires",groups=['DC','Myeloids'],
    spatial_key='spatial',scale_key="tissue_hires_scalef",library_id= f"{DSname}_hires_image", enforce_pixel_spot_size=False, img_alpha=.2)

# 3) Tumor

In [ ]:
obstomap = "spacexr"
groups = [i for i in adata.obs[obstomap].unique().tolist() if i != "Multiplets" and i != "Negative"]
print(groups)
plot_spatial_obs(
    adata, obs=obstomap, width=5, dpi=100,dotscale=.8 ,img_key="hires",groups=['Tumorcells'],
    spatial_key='spatial',scale_key="tissue_hires_scalef",library_id= f"{DSname}_hires_image", enforce_pixel_spot_size=False, img_alpha=.2)

# 4) CNS resident

In [ ]:
obstomap = "spacexr"
groups = [i for i in adata.obs[obstomap].unique().tolist() if i != "Multiplets" and i != "Negative"]
print(groups)
plot_spatial_obs(
    adata, obs=obstomap, width=5, dpi=100,dotscale=.8 ,img_key="hires",groups=['Oligodendrocytes',"Astrocytes","Neurons"],
    spatial_key='spatial',scale_key="tissue_hires_scalef",library_id= f"{DSname}_hires_image", enforce_pixel_spot_size=False, img_alpha=.2)

In [ ]:
adata

# Domains detection

# Prepare banksy

In [ ]:
sc.set_figure_params(facecolor="white", figsize=(8, 8))

adata.var_names_make_unique()
adata.var["mt"] = adata.var_names.str.startswith("MT-")

# Calulates QC metrics and put them in place to the adata object
sc.pp.calculate_qc_metrics(adata, 
                           qc_vars=["mt"], 
                           log1p=True, 
                           inplace=True)
from banksy_utils.plot_utils import plot_qc_hist, plot_cell_positions

# bin options for fomratting histograms
# Here, we set 'auto' for 1st figure, 80 bins for 2nd figure. and so on
hist_bin_options = ['auto', 80, 80, 100]

plot_qc_hist(adata, 
         total_counts_cutoff=150, 
         n_genes_high_cutoff=2500, 
         n_genes_low_cutoff=800,
         bin_options = hist_bin_options)

In [ ]:
from banksy_utils.filter_utils import filter_cells

# Filter cells with each respective filters
adata = filter_cells(adata, 
             min_count=10, 
             max_count=600, 
             MT_filter=20, 
             gene_filter=10)

In [ ]:
from banksy_utils.filter_utils import normalize_total, filter_hvg, print_max_min

import anndata
# Normalizes the anndata dataset
#adata = normalize_total(adata)

# JVC detection

In [ ]:
tag = "Domain"

SPneigh = 16
TXneighb = 50
npcs = 20

################# Free memory

print("Releasing memory")
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

##################



rsc.get.anndata_to_GPU(adata)
rsc.pp.filter_genes(adata, min_cells=10)
rsc.pp.normalize_total(adata)
rsc.pp.log1p(adata)
adata.layers["logCPU"]  = adata.X.get() # moves `.X` back to the CPU


# SVGs
sq.gr.spatial_neighbors(adata, coord_type="generic", n_neighs=SPneigh)
sq.gr.spatial_autocorr(adata, mode="moran", layer="logCPU")

SVGs = adata.uns["moranI"][adata.uns["moranI"]["pval_norm_fdr_bh"] < 0.01].dropna().head(2000).index.tolist()
len(SVGs)

# 2) pre-triku Neighbs
print("Scale1")
rsc.pp.scale(adata, zero_center=False, max_value=10)
rsc.pp.pca(adata,random_state=SEED, svd_solver="covariance_eigh")
sc.pl.pca_variance_ratio(adata)
print("neighbs1")
rsc.pp.neighbors(adata,   algorithm="brute", n_neighbors=TXneighb, n_pcs=npcs, random_state=SEED)

# 2) pre-triku Neighbs
print("Scale1")
rsc.pp.scale(adata, zero_center=False, max_value=10)
rsc.pp.pca(adata,random_state=SEED, svd_solver="covariance_eigh")
sc.pl.pca_variance_ratio(adata)
print("neighbs1")
rsc.pp.neighbors(adata,   algorithm="brute", n_neighbors=TXneighb, n_pcs=npcs, random_state=SEED)


# 3) Triku Neighbs
print("Triku")
adata.X = adata.layers["logCPU"].copy()
tk.tl.triku(adata, n_features=2000)

JointVGs = list(set(adata.var_names[adata.var["highly_variable"]].tolist()).union(set(SVGs)))
adata.var["jVGs"] = adata.var_names.isin(JointVGs)
jvgglen = adata.var["jVGs"].sum()
print(f"Total jVGs: {jvgglen}")


In [ ]:
rsc.get.anndata_to_CPU(adata)
adata.var["highly_variable"] = adata.var_names.isin(JointVGs)


In [ ]:
adata.X = adata.layers["counts"].copy()
del adata.layers["logCPU"]

In [ ]:
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

In [ ]:
adata_allgenes = adata.copy()
adata = adata[:,adata.var["highly_variable"]].copy()

## Start bansky pipeline

In [ ]:
from banksy_utils.load_data import load_adata, display_adata
adata.obs["xcoord"] = adata.obsm["FULLRESspatial_microns"][:,0]
adata.obs["ycoord"] = adata.obsm["FULLRESspatial_microns"][:,1]
display_adata(adata)


In [ ]:
from banksy.main import median_dist_to_nearest_neighbour
coord_keys = ('xcoord', 'ycoord', 'FULLRESspatial_microns')
# set params
# ==========
plot_graph_weights = True
k_geom = 25 # number of spatial neighbours
max_m = 1 # use both mean and AFT
nbr_weight_decay = "scaled_gaussian" # can also choose "reciprocal", "uniform" or "ranked"

# Find median distance to closest neighbours
nbrs = median_dist_to_nearest_neighbour(adata, key = coord_keys[2])

In [ ]:
from banksy.initialize_banksy import initialize_banksy
banksy_dict = initialize_banksy(
    adata,
    coord_keys,
    k_geom,
    nbr_weight_decay=nbr_weight_decay,
    max_m=max_m,
    plt_edge_hist=True,
    plt_nbr_weights=True,
    plt_agf_angles=False, # takes long time to plot
    plt_theta=True,
)

In [ ]:
from banksy.embed_banksy import generate_banksy_matrix

# The following are the main hyperparameters for BANKSY
# -----------------------------------------------------
resolutions = [0.4]  # clustering resolution for UMAP
pca_dims = [10]  # Dimensionality in which PCA reduces to
lambda_list = [0.4, 0.8]  # list of lambda parameters


banksy_dict, banksy_matrix = generate_banksy_matrix(adata,
                                                    banksy_dict,
                                                    lambda_list,
                                                    max_m)

In [ ]:
from banksy_utils.umap_pca import pca_umap

pca_umap(banksy_dict,
         pca_dims = pca_dims,
         add_umap = True,
         plt_remaining_var = False,
         )

# Port Back SNN graph to adata



In [ ]:
for lambdaParam in lambda_list:
    from banksy.main import LeidenPartition as BanksyPartition
    partitioner = BanksyPartition(input_space = banksy_dict["scaled_gaussian"][lambdaParam]["adata"].obsm["reduced_pc_10"],num_nn = 50)
    assert np.array_equal(adata_allgenes.obs_names.to_numpy(), banksy_dict["scaled_gaussian"][lambdaParam]["adata"].obs_names.to_numpy())
    W = partitioner.snn_weighted.tocsr().copy()
    W.setdiag(0)
    W.eliminate_zeros()
    
    # BANKSY’s kNN-derived graph is effectively directed; Scanpy/Leiden usually behaves better with symmetric adjacency.
    # This also better mimics BANKSY’s undirected igraph graph built from directed edges.
    W = (W + W.T)   # or: W = W.maximum(W.T)
    adata_allgenes.obsp[f"Banksy_weighted_snn_lambda{lambdaParam}"] = W
    adata_allgenes.uns[f"Banksy_weighted_snn_lambda{lambdaParam}"] = {}
    adata_allgenes.uns[f"Banksy_weighted_snn_lambda{lambdaParam}"]["connectivities_key"] = f"Banksy_weighted_snn_lambda{lambdaParam}"
    adata_allgenes.uns[f"Banksy_weighted_snn_lambda{lambdaParam}"]["distances_key"] = f"Banksy_weighted_snn_lambda{lambdaParam}"
    adata_allgenes.uns[f"Banksy_weighted_snn_lambda{lambdaParam}"]["params"] = 50
    adata_allgenes.uns[f"Banksy_weighted_snn_lambda{lambdaParam}"]
    adata_allgenes.obsm[f"Banksy_PCA_lambda{lambdaParam}"] = banksy_dict["scaled_gaussian"][lambdaParam]["adata"].obsm["reduced_pc_10"]
    adata_allgenes.obsm[f"Banksy_UMAP_lambda{lambdaParam}"] = banksy_dict["scaled_gaussian"][lambdaParam]["adata"].obsm["reduced_pc_10_umap"]
    start, stop, step = 0.2, 0.9, 0.1
    n = int(round((stop - start) / step)) + 1
    resolutions = np.linspace(start, stop, n)
    for i in resolutions:
        i = np.round(i,1)
        print(f"Partitioning with resolution {i}")
        sc.tl.leiden(
            adata_allgenes,
            neighbors_key = f"Banksy_weighted_snn_lambda{lambdaParam}",      # <-- uses THIS matrix :contentReference[oaicite:1]{index=1}
            directed=False,       # <-- important here :contentReference[oaicite:2]{index=2}
            use_weights=True,     # default True, but explicit is nice :contentReference[oaicite:3]{index=3}
            flavor="igraph",   # match BANKSY (your partitioner uses leidenalg.find_partition)
            resolution=i,       # use same scale as what “works well” for you
            random_state=1234,
            n_iterations=-1,
            key_added=f"Banksy_{str(i)}_lambda{lambdaParam}_partition")
    

In [ ]:
adata_allgenes.write_h5ad(f"./Banksy_{DS}.tmp.h5ad")

# Also partition with banksy native method

In [ ]:
from banksy.cluster_methods import run_Leiden_partition

results_df, max_num_labels = run_Leiden_partition(
    banksy_dict,
    resolutions,
    num_nn = 50, # k_expr: number of neighbours in expression (BANKSY embedding or non-spatial) space
    num_iterations = -1, # run to convergenece
    partition_seed = 1,
    match_labels = True,
)

In [ ]:
for lambdaParam in lambda_list:
    for res in resolutions:
        adata_allgenes.obs[f"scaled_gaussian_pc10_nc{lambdaParam}0_r{res}0"] = [str(kl) for kl in results_df.loc[f"scaled_gaussian_pc10_nc{lambdaParam}0_r{res}0","labels"].dense]
    

In [ ]:
adata_allgenes.write_h5ad(f"./Banksy_{DS}.tmp.h5ad")

# 1 Lets plot resoluts with Banksy native partitioner

In [ ]:
from banksy.plot_banksy import plot_results

c_map =  'tab20' # specify color map
weights_graph =  banksy_dict['scaled_gaussian']['weights'][0]

plot_results(
    results_df,
    weights_graph,
    c_map,
    match_labels = True,
    coord_keys = coord_keys,
    max_num_labels  =  max_num_labels, 
    save_path = "./tmp.png",
    save_fig = False
)

# Fast check the correct labels mapping

In [ ]:
plot_spatial_obs(
    adata_allgenes, obs="scaled_gaussian_pc10_nc0.80_r0.30", width=7, dpi=150,dotscale=1.5 ,img_key="hires",marker='o',
    spatial_key='spatial',scale_key="tissue_hires_scalef",library_id= "B19-25653_hires_image", enforce_pixel_spot_size=False, img_alpha=.4,  legend_kwargs={"fontsize":10})

# Now plot domains stability

In [ ]:
import os
import pandas as pd
import scanpy as sc
from tqdm import tqdm
import plotly.express as px
import pandas as pd
import matplotlib.colors as mcolors
import numpy as np
import plotly.express as px
import pandas as pd
import matplotlib.colors as mcolors
import scanpy.external as sce
from matplotlib.colors import LinearSegmentedColormap, to_hex


def plotSankey(obs, covs=[], save=None, show=True, method=None):
    """
    Create a Sankey-style parallel categories plot.

    Args:
        obs (pd.DataFrame): Input observation DataFrame.
        group (str): Group identifier for labeling.
        covs (list): List of columns to use for dimensions.
        save (str or None): File path to save the plot, or None to disable saving.
        show (bool): Whether to display the plot.
        method (str or None): Method for determining color logic ("consensus" or other).

    """
    # Validate input dimensions
    if not all(col in obs.columns for col in covs):
        raise ValueError("One or more dimensions in 'covs' are not valid columns in the provided DataFrame.")

    # Create a DataFrame with the selected covariates
    SankeyDF = obs[covs]

    if method == ["consensus"] or len(method) > 1:
        # Dynamically identify the columns to evaluate
        consensus_columns = [col for col in covs if "Labels" in col]

        # Calculate agreement levels
        SankeyDF["agreement"] = SankeyDF[consensus_columns].apply(
            lambda row: row.value_counts().max() if len(row.value_counts()) > 0 else 0, axis=1
        )

        # Remove "1" as it is not meaningful
        SankeyDF["agreement"] = SankeyDF["agreement"].replace(1, 0)

        # Total number of tools
        num_tools = len(consensus_columns)

        # Agreement levels: [0, 2, ..., num_tools]
        levels = [0] + list(range(2, num_tools + 1))  # Skip "1" as it's not meaningful

        # Generate equally spaced colors from red to green
        cmap = LinearSegmentedColormap.from_list("red_purple_blue", ["#e74c3c", "#9b59b6", "#2980b9"], N=len(levels))
        agreement_colors = {level: to_hex(cmap(i / (len(levels) - 1))) for i, level in enumerate(levels)}

        # Map agreement levels to colors
        SankeyDF["color"] = SankeyDF["agreement"].map(agreement_colors)

        # Define legend dynamically
        colorMap = {f"{level} Consistent Calls": agreement_colors[level] for level in levels}


    else:
        # Generate distinguishable colors using matplotlib's CSS color names
        distinct_colors = list(mcolors.TABLEAU_COLORS.values())
        unique_elements = SankeyDF[covs[0]].unique()
        colorMap = {element: distinct_colors[i % len(distinct_colors)] for i, element in enumerate(unique_elements)}
        SankeyDF["color"] = SankeyDF[covs[0]].replace(colorMap)

    # Create the parallel categories plot
    fig = px.parallel_categories(SankeyDF,
                                  dimensions=covs,
                                  color="color",
                                  width=1200, height=700)

    # Remove the colorbar
    fig.update_layout(coloraxis_showscale=False)

    # Adjust margins
    fig.update_layout(margin=dict(l=100, r=300, t=50, b=50))

    # Add legend
    x_start = 1.06
    y_start = 0.9
    rect_height = 0.05
    for i, (label, color) in enumerate(colorMap.items()):
        y_pos = y_start - i * rect_height
        fig.add_shape(type="rect",
                      xref="paper", yref="paper",
                      x0=x_start, y0=y_pos, x1=x_start + 0.03, y1=y_pos + rect_height,
                      line=dict(color='black'), fillcolor=color)
        fig.add_annotation(x=x_start + 0.035, y=y_pos + rect_height / 2,
                           xref="paper", yref="paper",
                           text=label, showarrow=False, font=dict(size=12),
                           xanchor="left", yanchor="middle")


    # Save the figure if save path is provided
    if save is not None:
        fig.write_image(save)
        print(f"Figure saved to {save}")

    # Display the figure if show is True
    if show:
        fig.show()


plotSankey(adata_allgenes.obs, covs=np.sort([c for c in adata_allgenes.obs if c.startswith("Banksy_0.") and c.endswith("_partition")]), save=None, show=True, method=[])

In [ ]:
sc.tl.pca(adata_allgenes, mask_var="triku_highly_variable")

# Check variance on TX pca

In [ ]:
plotResiduals(adata_allgenes, npcs=5, max_points=10000, covToTest=[c for c in adata_allgenes.obs if c.startswith("Banksy_0.") and c.endswith("_partition")])

In [ ]:
tag = "Banksy_0.7_partition"
variable1 = "spacexr"
variable2 = tag

# ----------------------------
# Settings: choose sort target
# ----------------------------
# Pick the spacexr category whose fraction you want to sort Banksy_domain by:
target_spacexr = "Tumorcells"   # e.g. "3" or "Astrocytes" etc.
sort_descending = True                     # True: highest first, False: lowest first

# ----------------------------
# Prep data + palettes
# ----------------------------
plotDF = adata_allgenes.obs[adata_allgenes.obs[variable2].notna()].copy()

assign_palette_topn_then_random(
    adata_allgenes,
    obs_key="spacexr",
    palette_name="tab20",
    random_seed=123
)
assign_palette_topn_then_random(
    adata_allgenes,
    obs_key=variable2,
    palette_name="Paired",
    random_seed=123
)

plotDF[variable1] = plotDF[variable1].astype(str)
plotDF[variable2] = plotDF[variable2].astype(str)

# ----------------------------
# Compute compositions
# ----------------------------
compositions = pd.DataFrame(plotDF.groupby([variable1, variable2]).size())
compositions = compositions.reset_index().rename(columns={0: "number_of_cells"})
compositions["clusterTotal"] = compositions.groupby(variable2)["number_of_cells"].transform("sum")
compositions["CellsFraction"] = compositions["number_of_cells"] / compositions["clusterTotal"]

# ----------------------------
# Get palette from adata.uns (keys as str)
# ----------------------------
cats = list(adata_allgenes.obs[variable1].cat.categories)
colors = adata_allgenes.uns[f"{variable1}_colors"]
cats_str = [str(c) for c in cats]
palette = {c: col for c, col in zip(cats_str, colors)}

# Keep hue_order only for categories present in the data
present = compositions[variable1].unique().tolist()
hue_order = [c for c in cats_str if c in present]

# ----------------------------
# Sort x-axis (Banksy_domain) by one spacexr fraction
# ----------------------------
piv = compositions.pivot_table(
    index=variable2, columns=variable1, values="CellsFraction", fill_value=0
)

if target_spacexr not in piv.columns:
    raise ValueError(
        f"target_spacexr='{target_spacexr}' not found in {variable1} categories.\n"
        f"Available: {list(piv.columns)}"
    )

x_order = piv[target_spacexr].sort_values(ascending=not sort_descending).index.tolist()

# Force the order on the plotting dataframe
compositions[variable2] = pd.Categorical(compositions[variable2], categories=x_order, ordered=True)

# ----------------------------
# Plot
# ----------------------------
fig, ax = plt.subplots(1, 1, figsize=(20, 5), dpi=100)
ax = sns.histplot(
    compositions,
    x=variable2,
    hue=variable1,
    weights="CellsFraction",
    multiple="stack",
    shrink=0.8,
    edgecolor="none",   # <- remove white borders
    palette=palette,
    hue_order=hue_order
)

ax.set_ylabel("CellsFraction")
ax.set(ylim=(0, 1))
sns.despine()

ax.grid(False)
ax.tick_params(axis="x", rotation=90)

legend = ax.get_legend()
if legend is not None:
    legend.set_bbox_to_anchor((1, 1))


In [ ]:
plot_spatial_obs(
    adata_allgenes, obs=tag, width=7, dpi=150,dotscale=1.5 ,img_key="hires",marker='o',
    spatial_key='spatial',scale_key="tissue_hires_scalef",library_id= "B19-25653_hires_image", enforce_pixel_spot_size=False, img_alpha=.4,  legend_kwargs={"fontsize":10})

In [ ]:
sc.pl.embedding(adata_allgenes, basis="Banksy_UMAP", color=tag,size=10, legend_loc="on data")

In [ ]:

for i in x_order:
    plot_spatial_obs(
        adata_allgenes, obs=tag, width=7, dpi=150,dotscale=1.5 ,img_key="hires",marker='o',groups=[i],
        spatial_key='spatial',scale_key="tissue_hires_scalef",library_id= "B19-25653_hires_image", enforce_pixel_spot_size=False, img_alpha=.4,  legend_kwargs={"fontsize":10})

    